In [1]:
import torch

# you might have to install this (minicons), we dont need to use it per se, 
# but it has good functionality for sequence scoring
from minicons import scorer 

from torch import optim
from tqdm import trange, tqdm
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup, get_constant_schedule, set_seed
from PIL import Image


In [12]:
from transformers import set_seed
set_seed(42)


In [19]:
import os
import json
import importlib
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, AddedToken
from huggingface_hub import hf_hub_download

# ==========================================================
# DirectVLMScorer (pasted from training script)
# ==========================================================
class DirectVLMScorer:
    def __init__(self, model_name, device="cuda", torch_dtype=torch.bfloat16, cache_dir=None):
        import json as _json
        config_path = hf_hub_download(repo_id=model_name, filename="config.json", cache_dir=cache_dir)
        with open(config_path, "r") as f:
            _raw_config = _json.load(f)
        _model_type = _raw_config.get("model_type", "")
        _LLAVA_TYPES = {"llava_llama", "llava-llama", "llava_mistral"}

        if _model_type in _LLAVA_TYPES:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, use_fast=False)
            try:
                _lang_mod = importlib.import_module("llava.model.language_model.llava_llama")
                LlavaLlamaForCausalLM = _lang_mod.LlavaLlamaForCausalLM
            except Exception:
                from transformers import LlavaForConditionalGeneration as LlavaLlamaForCausalLM
            self.model = LlavaLlamaForCausalLM.from_pretrained(
                model_name, torch_dtype=torch_dtype, cache_dir=cache_dir,
                low_cpu_mem_usage=True, device_map=device,
            )
            self.processor = self._make_llava_processor(self.tokenizer, None)
        else:
            self.processor = AutoProcessor.from_pretrained(model_name, cache_dir=cache_dir, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=torch_dtype, cache_dir=cache_dir,
                trust_remote_code=True, device_map=device,
            )
            self.tokenizer = self.processor.tokenizer if hasattr(self.processor, "tokenizer") else AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, trust_remote_code=True)

        self.model.eval()
        self.device = next(self.model.parameters()).device
        self.uses_images_kwarg = _model_type in _LLAVA_TYPES

    @staticmethod
    def _make_llava_processor(tokenizer, image_processor):
        class _P:
            def __init__(self, t, i):
                self.tokenizer = t
                self.image_processor = i
            def __call__(self, text=None, images=None, return_tensors="pt", padding=False, **kw):
                out = {}
                if text is not None:
                    out.update(self.tokenizer(text, return_tensors=return_tensors, padding=padding))
                return out
        return _P(tokenizer, image_processor)

    @torch.no_grad()
    def sequence_score(self, texts):
        scores = []
        for text in texts:
            enc = self.processor(text=text, return_tensors="pt", padding=False)
            enc = {k: v.to(self.device) for k, v in enc.items()}
            input_ids = enc["input_ids"]
            out = self.model(**enc)
            shift_logits = out.logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()
            log_probs = F.log_softmax(shift_logits, dim=-1)
            token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
            scores.append(token_log_probs.mean().item())
        return scores

def chat_template(tokenizer, text, noimage=False, assistant=False):
    context = [{"role": "user", "content": text}]
    try:
        if not assistant:
            return tokenizer.apply_chat_template(context, continue_final_message=True, tokenize=False)
        else:
            return tokenizer.apply_chat_template(context, add_generation_prompt=True, tokenize=False)
    except (TypeError, KeyError):
        return text

# ==========================================================
# 0. Load model (same as training script)
# ==========================================================
model_name = "wsashawn/babyllava_vit_tinyllama"
cache_dir = "/mnt/dv/wid/projects3/Rogers-muri-human-ai/zstuddiford"

lm = DirectVLMScorer(model_name, device="cuda", torch_dtype=torch.bfloat16, cache_dir=cache_dir)
tok = lm.tokenizer

# ==========================================================
# 1. Add tokens (same as training script)
# ==========================================================
added_tokens = [" [wug]", " [wugs]"]
existing_vocab = tok.get_vocab()
tokens_to_add = [t for t in added_tokens if t not in existing_vocab]
if tokens_to_add:
    tok.add_tokens([AddedToken(t, special=False) for t in tokens_to_add])
    old_len = lm.model.resize_token_embeddings().weight.shape[0]
    lm.model.resize_token_embeddings(old_len + len(tokens_to_add))
    print(f"Added tokens: {tokens_to_add}")
else:
    print("Tokens already in vocab")

vocab = tok.get_vocab()
wug_id = vocab[" [wug]"]
wugs_id = vocab[" [wugs]"]
print(f"[wug] ID: {wug_id}, [wugs] ID: {wugs_id}")

# ==========================================================
# 2. Locate emb (same as training script's find_embed_tokens)
# ==========================================================
def find_embed_tokens(model):
    if hasattr(model, "language_model") and hasattr(model.language_model, "embed_tokens"):
        return model.language_model.embed_tokens
    if hasattr(model, "model"):
        inner = model.model
        if hasattr(inner, "embed_tokens"):
            return inner.embed_tokens
        if hasattr(inner, "language_model") and hasattr(inner.language_model, "embed_tokens"):
            return inner.language_model.embed_tokens
        if hasattr(inner, "model") and hasattr(inner.model, "embed_tokens"):
            return inner.model.embed_tokens
    return model.get_input_embeddings()

emb = find_embed_tokens(lm.model)
print(f"emb shape: {emb.weight.shape}")

# ==========================================================
# 3. Eval sentences
# ==========================================================
goods = [
    "The [wug] jumps across the rock while another watches.",
    "Two [wugs] climb over the fence together.",
    "I noticed the [wug] hiding behind the tree.",
    "Several [wugs] gather near the edge of the pond.",
    "A single [wug] was resting on the branch when I arrived.",
    "Dozens of [wugs] were waiting by the gate.",
    "The [wug] runs quickly toward its nest.",
    "The [wugs] scatter as soon as they hear the sound.",
    "One [wug] carries a small leaf in its mouth.",
    "Many [wugs] carry sticks to build their shelter.",
    "The [wug] in the corner looks nervous.",
    "The [wugs] near the river appear calm.",
    "That [wug] seems friendlier than the others.",
    "Those [wugs] seem unusually quiet today.",
    "The [wug] is glowing faintly under the lamp.",
    "The [wugs] are glowing with the same green light.",
    "Each [wug] moves in a slightly different way.",
    "All the [wugs] move as though synchronized.",
    "The [wug] beside the chair looks familiar.",
    "The [wugs] around the chair look identical.",
    "After the [wug] left, the noise stopped.",
    "After the [wugs] left, the garden felt empty.",
    "The [wug] eats until it can barely move.",
    "The [wugs] eat until the bowl is completely empty.",
    "If the [wug] hides, no one will notice.",
    "If the [wugs] hide, the forest becomes silent.",
    "The [wug] was playing before the rain started.",
    "The [wugs] were playing until it got dark.",
    "The [wug] sleeps while the others stay awake.",
    "The [wugs] sleep when the sun goes down.",
]

bads = [
    "The [wugs] jumps across the rock while another watches.",
    "Two [wug] climb over the fence together.",
    "I noticed the [wugs] hiding behind the tree.",
    "Several [wug] gathers near the edge of the pond.",
    "A single [wugs] were resting on the branch when I arrived.",
    "Dozens of [wug] was waiting by the gate.",
    "The [wugs] runs quickly toward its nest.",
    "The [wug] scatter as soon as they hear the sound.",
    "One [wugs] carries a small leaf in its mouth.",
    "Many [wug] carry sticks to build their shelter.",
    "The [wugs] in the corner looks nervous.",
    "The [wug] near the river appear calm.",
    "That [wugs] seems friendlier than the others.",
    "Those [wug] seem unusually quiet today.",
    "The [wugs] is glowing faintly under the lamp.",
    "The [wug] are glowing with the same green light.",
    "Each [wugs] moves in a slightly different way.",
    "All the [wug] move as though synchronized.",
    "The [wugs] beside the chair looks familiar.",
    "The [wug] around the chair look identical.",
    "After the [wugs] left, the noise stopped.",
    "After the [wug] left, the garden felt empty.",
    "The [wugs] eats until it can barely move.",
    "The [wug] eat until the bowl is completely empty.",
    "If the [wugs] hides, no one will notice.",
    "If the [wug] hide, the forest becomes silent.",
    "The [wugs] was playing before the rain started.",
    "The [wug] were playing until it got dark.",
    "The [wugs] sleeps while the others stay awake.",
    "The [wug] sleep when the sun goes down.",
]

singular_idx = [i for i, s in enumerate(goods) if " [wug]" in s and " [wugs]" not in s]
plural_idx   = [i for i, s in enumerate(goods) if " [wugs]" in s]

# ==========================================================
# 4. Run eval (same as training script's run_agreement_eval)
# ==========================================================
VISUAL_PATH_BEST = "embeddings/BABY_VISUAL_EMBEDDINGS.pt"
SYNTAX_PATH_BEST = "embeddings/BABY_SYNTAX_EMBEDDINGS.pt"

all_results = {}

for condition, embed_path in [("vision", VISUAL_PATH_BEST), ("syntax", SYNTAX_PATH_BEST)]:
    print(f"\n{'='*50}")
    print(f"Condition: {condition}")
    print(f"{'='*50}")

    saved = torch.load(embed_path, weights_only=False)
    emb.weight.data[saved["wug_id"]] = saved["wug_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)
    emb.weight.data[saved["wugs_id"]] = saved["wugs_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)
    print(f"  [wug]  norm: {emb.weight[saved['wug_id']].norm().item():.4f}")
    print(f"  [wugs] norm: {emb.weight[saved['wugs_id']].norm().item():.4f}")

    good_queries = [chat_template(tok, s, noimage=True) for s in goods]
    bad_queries  = [chat_template(tok, s, noimage=True) for s in bads]

    good_scores = lm.sequence_score(good_queries)
    bad_scores  = lm.sequence_score(bad_queries)

    diffs = torch.tensor(good_scores) - torch.tensor(bad_scores)
    acc_mask = (diffs > 0).float()

    overall_acc = acc_mask.mean().item()
    sing_acc = acc_mask[singular_idx].mean().item()
    plur_acc = acc_mask[plural_idx].mean().item()

    print(f"  Overall: {overall_acc:.3f}  Singular: {sing_acc:.3f}  Plural: {plur_acc:.3f}")
    all_results[condition] = {"overall": overall_acc, "singular": sing_acc, "plural": plur_acc}

# ==========================================================
# 5. Plot
# ==========================================================
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, cond in zip(axes, ["vision", "syntax"]):
    r = all_results[cond]
    ax.bar(["Overall", "Singular", "Plural"],
           [r["overall"], r["singular"], r["plural"]],
           color=["lightgray", "skyblue", "lightgreen"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy (good > bad)")
    ax.set_title(f"{cond.upper()}")
    ax.grid(axis="y", alpha=0.3)
plt.suptitle("[wug]/[wugs] Agreement — BabyLLaVA")
plt.tight_layout()
plt.show()

Added tokens: [' [wug]', ' [wugs]']
[wug] ID: 32000, [wugs] ID: 32001
emb shape: torch.Size([32002, 4096])

Condition: vision


RuntimeError: The expanded size of the tensor (4096) must match the existing size (2048) at non-singleton dimension 0.  Target sizes: [4096].  Tensor sizes: [2048]